In [1]:
import os
import re
import pandas as pd

def parse_restraint(restraint):
    try:
        if not isinstance(restraint, str):
            return None, None, None, None

        restraint = restraint.strip()
        if restraint.count('-') != 1:
            return None, None, None, None

        part1, part2 = restraint.split('-')

        def split_part(part):
            match = re.match(r'^(\d+)(.+)', part)
            if match:
                return match.groups()
            else:
                return None, part

        res1, atom1 = split_part(part1)
        res2, atom2 = split_part(part2)

        if res2 is None:
            res2 = res1

        return res1, atom1, res2, atom2

    except Exception as e:
        print(f"Error: {e}")
        return None, None, None, None

def convert_single_restraint_csv_to_itp(csv_file, itp_path, mol_name):
    df = pd.read_csv(csv_file)
    df.columns = ['restraint', 'intensity']

    intensity_priority = {'weak': 0, 'medium': 1, 'strong': 2}
    bond_map = {
        'strong': '0.00 0.25 0.35 1000  0.00 0.25 0.35 0',
        'medium': '0.00 0.30 0.40 1000  0.00 0.30 0.40 0',
        'weak':   '0.00 0.40 0.55 1000  0.00 0.30 0.40 0'
    }

    cgnr_lookup = {
        'H02': {
            'α11': ['H12', 'H13'],
            'α2': ['H3'],
            'β3': ['H4', 'H5', 'H6']
        },
        'R02': {
            'α11': ['H10', 'H11'],
            'α2': ['H1'],
            'β3': ['H2', 'H3', 'H4']
        },
        'NH2': {
            'NH1': ['H1', 'H2']
        }
    }

    parsed_restraints = []
    for i, row in df.iterrows():
        restraint = row['restraint']
        intensity = row['intensity']

        if pd.isna(restraint) or pd.isna(intensity):
            continue

        res1, atom1, res2, atom2 = parse_restraint(str(restraint))

        if None not in (res1, atom1, res2, atom2):
            parsed_restraints.append({
                'res1': res1,
                'atom1': atom1.replace('α12', 'α11').replace('NH2', 'NH1'),
                'res2': res2,
                'atom2': atom2.replace('α12', 'α11').replace('NH2', 'NH1'),
                'intensity': intensity
            })

    df_parsed = pd.DataFrame(parsed_restraints)
    df_parsed['intensity_score'] = df_parsed['intensity'].map(intensity_priority)
    df_sorted = df_parsed.sort_values('intensity_score')
    df_merged = df_sorted.drop_duplicates(subset=['res1', 'atom1', 'res2', 'atom2'], keep='first')

    itp_file = f"{mol_name}.itp"
    atoms_file = os.path.join(itp_path, itp_file)

    if not os.path.exists(atoms_file):
        print(f"❌ Missing .itp file for {mol_name}: {atoms_file}")
        return

    # Parse atom section from .itp
    start_tag = "[ atoms ]"
    block = []
    in_block = False

    with open(atoms_file, 'r') as file:
        for line in file:
            stripped = line.strip()
            if stripped.startswith('[') and stripped.endswith(']'):
                if stripped.lower() == start_tag:
                    in_block = True
                    continue
                elif in_block:
                    break
            if in_block and stripped and not stripped.startswith(';'):
                block.append(line.strip())

    atom_records = []
    for line in block:
        line = line.split(';')[0].strip()
        parts = line.split()
        if len(parts) >= 6:
            nr, atom_type, resi, res, atom, cgnr = parts[:6]
            atom_records.append([nr, atom_type, resi, res, atom, cgnr])

    df_atoms = pd.DataFrame(atom_records, columns=['nr', 'type', 'resi', 'res', 'atom', 'cgnr'])

    restraint_list = []
    for i, row in df_merged.iterrows():
        res1_i = int(row['res1'])
        res2_i = int(row['res2'])
        atom1 = row['atom1']
        atom2 = row['atom2']
        intensity = row['intensity']

        if atom1 == 'NH1':
            res1_i += 1
        if atom2 == 'NH1':
            res2_i += 1

        res1_i = str(res1_i)
        res2_i = str(res2_i)

        match1 = df_atoms[df_atoms['resi'] == res1_i]
        match2 = df_atoms[df_atoms['resi'] == res2_i]

        if match1.empty or match2.empty:
            continue

        res1_res_name = match1['res'].values[0]
        res2_res_name = match2['res'].values[0]

        cgnr_labels1 = cgnr_lookup.get(res1_res_name, {}).get(atom1, [])
        cgnr_labels2 = cgnr_lookup.get(res2_res_name, {}).get(atom2, [])

        cgnr_nums_1 = df_atoms[(df_atoms['resi'] == res1_i) & (df_atoms['atom'].isin(cgnr_labels1))]['cgnr'].tolist()
        cgnr_nums_2 = df_atoms[(df_atoms['resi'] == res2_i) & (df_atoms['atom'].isin(cgnr_labels2))]['cgnr'].tolist()

        for c1 in cgnr_nums_1:
            for c2 in cgnr_nums_2:
                restraint_list.append({
                    'atom1': c1,
                    'atom2': c2,
                    'Intensity': intensity
                })

    df_ready = pd.DataFrame(restraint_list)

    restraint_lines = []
    restraint_lines.append(";  ai   aj  funct  r0_A  r1_A  r2_A  k_A  r0_B  r1_B  r2_B  k_B  flat-bottom parameters")
    for i, row in df_ready.iterrows():
        ai = row['atom1']
        aj = row['atom2']
        intensity = row['Intensity'].lower()
        bond = bond_map.get(intensity)
        if bond:
            restraint_line = f"  {ai}  {aj}  {bond}"
            restraint_lines.append(restraint_line)

    # Insert into .itp
    start_tag = '[ bonds ]'
    new_itp_file = itp_file.replace('.itp', '_re.itp')
    new_itp_path = os.path.join(itp_path, new_itp_file)

    with open(atoms_file, 'r') as f:
        lines = f.readlines()

    output_lines = []
    in_bonds_section = False
    bonds_inserted = False

    for line in lines:
        stripped = line.strip()
        output_lines.append(line)

        if stripped == start_tag:
            in_bonds_section = True
            continue

        if in_bonds_section and stripped.startswith('[') and stripped.endswith(']'):
            if not bonds_inserted:
                output_lines = output_lines[:-1]
                output_lines.extend([r + '\n' for r in restraint_lines])
                output_lines.append(line)
                bonds_inserted = True
            in_bonds_section = False
            continue

    if in_bonds_section and not bonds_inserted:
        output_lines[-1] = output_lines[-1].rstrip('\n')
        output_lines.extend([r + '\n' for r in restraint_lines])

    with open(new_itp_path, 'w') as f:
        f.writelines(output_lines)

    print(f"✅ Appended restraints to [ bonds ] section and saved as: {new_itp_path}")


In [3]:
convert_single_restraint_csv_to_itp('restraints_file/nspe_7_1_res.csv', 'itp', 'nspe_7_1')
convert_single_restraint_csv_to_itp('restraints_file/nspe_7_2_res.csv', 'itp', 'nspe_7_2')
convert_single_restraint_csv_to_itp('restraints_file/nspe_10_res.csv', 'itp', 'nspe_10')


✅ Appended restraints to [ bonds ] section and saved as: itp/nspe_7_1_re.itp
✅ Appended restraints to [ bonds ] section and saved as: itp/nspe_7_2_re.itp
✅ Appended restraints to [ bonds ] section and saved as: itp/nspe_10_re.itp
